In [ ]:
import pandas as pd
file_path = '/content/Synthetic News Dataset & Model Details - Synthetic News Dataset.csv'
df = pd.read_csv(file_path)

In [ ]:
df.columns = df.iloc[0]

In [ ]:
df = df[1:].reset_index(drop=True)

In [ ]:
df.head()

,Sr. No,Newspaper Name,Published Date,URL,Headline,Content,Human Summary,Category
0,1,The Hindu,2023-12-01,https://www.thehindu.com/news/national/sample-...,"""India Launches Chandrayaan-4 Successfully""","India successfully launched Chandrayaan-4, aim...",India launched Chandrayaan-4 to study the moon...,Science and Technology
1,2,Hindustan Times,2022-08-15,https://www.hindustantimes.com/india/sample-ne...,"""PM Announces Digital India 2.0 on Independenc...",The Prime Minister unveiled the Digital India ...,"PM launched Digital India 2.0, focusing on tec...",National News
2,3,Indian Express,2021-04-10,https://www.indianexpress.com/news/sample-news-3,"""Economic Growth Rebounds in Q1 2021""",India’s GDP showed a rebound in the first quar...,"India's Q1 2021 GDP rebounded, indicating a re...",Business and Finance
3,4,The Telegraph,2023-05-18,https://www.telegraphindia.com/nation/sample-n...,"""Cyclone Yaas Causes Widespread Damage in East...",Cyclone Yaas wreaked havoc in Odisha and West ...,Cyclone Yaas caused severe damage in Eastern I...,Environment
4,5,Deccan Chronicle,2020-10-05,https://www.deccanchronicle.com/nation/sample-...,"""Hyderabad Emerges as India’s Vaccine Hub""",Hyderabad became a central hub for COVID-19 va...,Hyderabad gained recognition as the COVID-19 v...,Health and Wellness


In [ ]:
import csv
import re
def clean_text_for_summarization(text):
    text = re.sub(r"\(ID #\d+\)", "", text)
    text = re.sub(r"\.{3,}", "", text)
    text = re.sub(r"[^\w\s.,!?;']", "", text)
    text = " ".join(text.split())
    text = text.replace("\n", " ").replace("\r", " ")
    text = text.replace('"', "").replace("'", "")
    return text.strip()

In [ ]:
df['Content'] = df['Content'].apply(clean_text_for_summarization)

In [ ]:
df['Human Summary'] = df['Human Summary'].apply(clean_text_for_summarization)

In [ ]:
df['Headline'] = df['Headline'].apply(clean_text_for_summarization)

In [ ]:
df['Category'] = df['Category'].apply(clean_text_for_summarization)

In [ ]:
new_df = df[['Content', 'Human Summary','Category','Headline']]

In [ ]:
new_df.head()

,Content,Human Summary
0,"India successfully launched Chandrayaan4, aimi...",India launched Chandrayaan4 to study the moons...
1,The Prime Minister unveiled the Digital India ...,"PM launched Digital India 2.0, focusing on tec..."
2,Indias GDP showed a rebound in the first quart...,"Indias Q1 2021 GDP rebounded, indicating a rec..."
3,Cyclone Yaas wreaked havoc in Odisha and West ...,Cyclone Yaas caused severe damage in Eastern I...
4,Hyderabad became a central hub for COVID19 vac...,Hyderabad gained recognition as the COVID19 va...


In [ ]:
!pip install transformers datasets rouge_score torch
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import PegasusTokenizer, PegasusForConditionalGeneration, Trainer, TrainingArguments, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.6 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=54db557151b5a59d322a291de0f4fae95875299c3e7bee6fefe3606ae7acfdda
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that 

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.4 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from transformers import PegasusTokenizer, PegasusForConditionalGeneration
from datasets import Dataset
from transformers import Trainer, TrainingArguments
import os
import evaluate

# Attention Mechanism for Word Level
class WordLevelAttention(nn.Module):
    def __init__(self, hidden_size):
        super(WordLevelAttention, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, mask):
        mask = mask.to(hidden_states.device)
        scores = self.attention(hidden_states).squeeze(-1)  # [batch_size, seq_len]
        scores = scores.masked_fill(~mask, float("-inf"))  # Mask padding tokens
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)  # [batch_size, seq_len, 1]
        context_vector = torch.sum(hidden_states * weights, dim=1)  # [batch_size, hidden_size]
        return context_vector, weights

# Attention Mechanism for Sentence Level
class SentenceLevelAttention(nn.Module):
    def __init__(self, hidden_size):
        super(SentenceLevelAttention, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, mask):
        mask = mask.to(hidden_states.device)
        scores = self.attention(hidden_states).squeeze(-1)  # [batch_size, num_sentences]
        scores = scores.masked_fill(~mask, float("-inf"))  # Mask padding sentences
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)  # [batch_size, num_sentences, 1]
        context_vector = torch.sum(hidden_states * weights, dim=1)  # [batch_size, hidden_size]
        return context_vector, weights

class HTAPegasus(nn.Module):
    def __init__(self, pegasus_model_name, hidden_size=None):
        super(HTAPegasus, self).__init__()
        self.tokenizer = PegasusTokenizer.from_pretrained(pegasus_model_name)
        self.pegasus = PegasusForConditionalGeneration.from_pretrained(pegasus_model_name)
        self.hidden_size = hidden_size or self.pegasus.config.d_model
        self.word_attention = WordLevelAttention(self.hidden_size)
        self.sentence_attention = SentenceLevelAttention(self.hidden_size)

    def forward(self, input_ids, attention_mask, labels=None):
        # Pass input_ids and attention_mask to the model's encoder
        output = self.pegasus.model.encoder(
            input_ids=input_ids.to(self.pegasus.device),
            attention_mask=attention_mask.to(self.pegasus.device),
        )
        hidden_states = output.last_hidden_state  # [batch_size, seq_len, hidden_size]

        # Use attention mechanisms on the hidden states
        mask = attention_mask.to(torch.bool).to(hidden_states.device)
        context_vector, _ = self.word_attention(hidden_states, mask)

        sentence_embeddings = [context_vector]
        sentence_masks = [torch.ones(context_vector.size(0), dtype=torch.bool).to(hidden_states.device)]

        sentence_embeddings_padded = nn.utils.rnn.pad_sequence(sentence_embeddings, batch_first=True)
        sentence_masks_padded = nn.utils.rnn.pad_sequence(sentence_masks, batch_first=True)

        # Sentence-level attention for document representation
        document_representation, _ = self.sentence_attention(sentence_embeddings_padded, sentence_masks_padded)

        # If labels are provided, calculate loss
        if labels is not None:
            loss = self.pegasus(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            return loss
        else:
            # If no labels are provided, return the document representation
            return document_representation
# Add the generate method
    def generate(self, input_ids, attention_mask=None, **kwargs):
        # Use the generate method from the Pegasus model
        return self.pegasus.generate(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
    def save_pretrained(self, output_dir):
        # Save the tokenizer
        self.tokenizer.save_pretrained(output_dir)

        # Save the Pegasus model
        self.pegasus.save_pretrained(output_dir)



In [ ]:
!pip install safetensors


In [ ]:
# Dataset conversion from pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(new_df)

# Split dataset into train and validation sets (80% train, 20% validation)
train_test_split = dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split["train"]
val_dataset = train_test_split["test"]

# Define tokenizer for input and target sequences
tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-xsum")

# Tokenization function
def tokenize_function(examples):
    # Concatenate 'Content', 'Category', and 'Headline' into a single string
    inputs = [f"Category: {cat} Headline: {head} Content: {content}"
              for cat, head, content in zip(examples["Category"], examples["Headline"], examples["Content"])]

    # Tokenize the concatenated input
    model_inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=512)

    # Tokenize the target (Human Summary)
    targets = tokenizer(
        examples["Human Summary"], truncation=True, padding="max_length", max_length=128
    )

    # Add the labels to the inputs for training
    model_inputs["labels"] = targets["input_ids"]
    return model_inputs


# Apply tokenization to train and validation datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

output_dir = '/content/saved_hta_pegasus'
import os
os.makedirs(output_dir, exist_ok=True)


# Disable W&B logging completely
os.environ["WANDB_DISABLED"] = "true"
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# Initialize HTAPegasus model
hta_model = HTAPegasus(pegasus_model_name="google/pegasus-xsum")
hta_model.to("cuda" if torch.cuda.is_available() else "cpu")  # Move model to GPU if available

# Define the training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/saved_hta_pegasus",
    per_device_train_batch_size=2,  # Reduce batch size for limited GPU memory
    gradient_accumulation_steps=8,  # Accumulate gradients to simulate a larger batch size
    num_train_epochs=3,  # Number of training epochs
    save_strategy="epoch",  # Save model at the end of each epoch
    evaluation_strategy="epoch",  # Evaluate the model at the end of each epoch
    logging_dir="/content/saved_hta_pegasus",  # Directory for logs
    logging_steps=25,  # Log every 50 steps
    save_total_limit=1,  # Keep only the latest checkpoint
    load_best_model_at_end=True,  # Automatically load the best model
    fp16=True,  # Enable mixed precision for faster training
    predict_with_generate=True,  # Generate summaries during evaluation
)

# Define the data collator
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=hta_model.tokenizer, model=hta_model.pegasus)

# Initialize the Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=hta_model.pegasus,  # Your Pegasus model
    args=training_args,  # Training arguments
    train_dataset=train_dataset,  # Training dataset
    eval_dataset=val_dataset,  # Evaluation dataset
    tokenizer=hta_model.tokenizer,  # Tokenizer
    data_collator=data_collator,  # Data collator
)

# Train the model
trainer.train()

# Initialize HTAPegasus model


# Save the model and tokenizer
output_dir = "/content/saved_hta_pegasus"
hta_model.save_pretrained(output_dir)

tokenizer = PegasusTokenizer.from_pretrained(output_dir)

hta_model = HTAPegasus(pegasus_model_name="google/pegasus-xsum")
hta_model.tokenizer = tokenizer  # Assign reloaded tokenizer
hta_model.pegasus = PegasusForConditionalGeneration.from_pretrained(output_dir)  # Assign reloaded Pegasus model

# hta_model = HTAPegasus.from_pretrained(output_dir)
# hta_model.tokenizer.save_pretrained(output_dir)
# hta_model.save_pretrained(output_dir)
# tokenizer = PegasusTokenizer.from_pretrained(output_dir)
# hta_model = HTAPegasus.from_pretrained(output_dir)
# # Reinitialize the HTAPegasus model with reloaded components
# hta_model = HTAPegasus(pegasus_model_name="google/pegasus-xsum")
# hta_model.tokenizer = tokenizer  # Assign reloaded tokenizer
# hta_model.pegasus = pegasus_model  # Assign reloaded Pegasus model
# Ensure the model is on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hta_model.to(device)



Map:   0%|          | 0/89 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-14-43b38f20aee4>:61: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
0,No log,4.819164
1,49.623300,4.586260
2,49.623300,4.461963


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 64, 'num_beams': 8, 'length_penalty': 0.6}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


HTAPegasus(
  (pegasus): PegasusForConditionalGeneration(
    (model): PegasusModel(
      (shared): Embedding(96103, 1024, padding_idx=0)
      (encoder): PegasusEncoder(
        (embed_tokens): Embedding(96103, 1024, padding_idx=0)
        (embed_positions): PegasusSinusoidalPositionalEmbedding(512, 1024)
        (layers): ModuleList(
          (0-15): 16 x PegasusEncoderLayer(
            (self_attn): PegasusAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (activation_fn): ReLU()
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (fc2): Linear(in_features=409

In [ ]:
import torch

# Ensure the model is on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hta_model.to(device)

# Generate summaries for validation set
predictions = []
references = []

for example in val_dataset:
    # Concatenate the Content, Category, and Headline for the input
    combined_input = f"Category: {example['Category']} Headline: {example['Headline']} Content: {example['Content']}"

    # Tokenize the combined input and move to the correct device
    inputs = tokenizer(
        combined_input, return_tensors="pt", truncation=True, padding="max_length", max_length=512
    )
    input_ids = inputs.input_ids.to(device)  # Move input_ids to the correct device
    attention_mask = inputs.attention_mask.to(device)  # Also move attention_mask if provided

    # Generate summary
    output = hta_model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=128,
        num_beams=5
    )

    # Decode the generated output
    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    predictions.append(pred)

    # Collect the ground truth reference
    references.append(example["Human Summary"])

In [ ]:
import evaluate

# Load the ROUGE metric
rouge = evaluate.load("rouge")

# Initialize lists to store individual example ROUGE scores
example_rouge_scores = []

# Loop over each example in the dataset and calculate ROUGE scores
for i, (pred, ref) in enumerate(zip(predictions, references)):
    # Compute ROUGE scores for each example
    example_rouge = rouge.compute(predictions=[pred], references=[ref])
    example_rouge_scores.append(example_rouge)

    # Display ROUGE scores for each example with index
    print(f"Example {i+1} ROUGE scores: {example_rouge}")

# Calculate average ROUGE scores across all examples
average_rouge_scores = {
    "rouge1": sum([score["rouge1"] for score in example_rouge_scores]) / len(example_rouge_scores),
    "rouge2": sum([score["rouge2"] for score in example_rouge_scores]) / len(example_rouge_scores),
    "rougeL": sum([score["rougeL"] for score in example_rouge_scores]) / len(example_rouge_scores)
}

# Display the final average ROUGE scores
print(f"Average ROUGE-1: {average_rouge_scores['rouge1']}")
print(f"Average ROUGE-2: {average_rouge_scores['rouge2']}")
print(f"Average ROUGE-L: {average_rouge_scores['rougeL']}")

Example 1 ROUGE scores: {'rouge1': 0.15267175572519084, 'rouge2': 0.015503875968992248, 'rougeL': 0.0916030534351145, 'rougeLsum': 0.0916030534351145}
Example 2 ROUGE scores: {'rouge1': 0.33587786259541985, 'rouge2': 0.248062015503876, 'rougeL': 0.3206106870229008, 'rougeLsum': 0.3206106870229008}
Example 3 ROUGE scores: {'rouge1': 0.5454545454545454, 'rouge2': 0.42666666666666664, 'rougeL': 0.5194805194805194, 'rougeLsum': 0.5194805194805194}
Example 4 ROUGE scores: {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'rougeLsum': 0.0}
Example 5 ROUGE scores: {'rouge1': 0.3013698630136986, 'rouge2': 0.09722222222222222, 'rougeL': 0.273972602739726, 'rougeLsum': 0.273972602739726}
Example 6 ROUGE scores: {'rouge1': 0.1553398058252427, 'rouge2': 0.039603960396039604, 'rougeL': 0.11650485436893203, 'rougeLsum': 0.11650485436893203}
Example 7 ROUGE scores: {'rouge1': 0.2909090909090909, 'rouge2': 0.12269938650306748, 'rougeL': 0.19393939393939394, 'rougeLsum': 0.19393939393939394}
Example 8 ROUG

In [ ]:
print(predictions[1])
print(references[1])

The Central Bureau of Investigation (CBI) on Thursday questioned three doctors of a private hospital in Kolkata in connection with the rape and murder of a junior doctor.
The CBI summoned five doctors and junior doctors from RG Kar Medical College in connection with the rape and murder of an onduty doctor on August 9. Three doctors, including former medical superintendent Dr. Sanjay Vashisth, were questioned, and further questioning is planned. The CBI also questioned a local police officer involved in the case.
